<a href="https://colab.research.google.com/github/letiBri/MaskArchitectureAnomaly_CourseProject/blob/main/STEP8_Ari.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Anomaly Segmentation Eomt vs ErfNet (Step 8)

In [1]:
!git clone https://github.com/letiBri/MaskArchitectureAnomaly_CourseProject.git
%cd MaskArchitectureAnomaly_CourseProject/eomt

fatal: destination path 'MaskArchitectureAnomaly_CourseProject' already exists and is not an empty directory.
/content/MaskArchitectureAnomaly_CourseProject/eomt


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
!pip install ood-metrics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 43.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you

In [17]:
!pip install -r requirements.txt

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.4/219.4 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 92.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 88.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 819.0/819.0 kB 62.3 MB/s 

In [3]:
import sys
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject')
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject/eval')
sys.path.append('/content/MaskArchitectureAnomaly_CourseProject/eomt')


In [4]:
import os
import cv2
import glob
import torch
import random
from PIL import Image
import numpy as np
from erfnet import ERFNet
import os.path as osp
from argparse import ArgumentParser
from ood_metrics import fpr_at_95_tpr, calc_metrics, plot_roc, plot_pr,plot_barcode
from sklearn.metrics import roc_auc_score, roc_curve, auc, precision_recall_curve, average_precision_score
from torchvision.transforms import Compose, Resize, ToTensor, Normalize

# Setup

In [5]:
import yaml
from lightning import seed_everything
import torch
from torch.nn import functional as F
from torch.amp.autocast_mode import autocast
import matplotlib.pyplot as plt
import numpy as np
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import RepositoryNotFoundError
import warnings
import importlib

seed_everything(0, verbose=False)

device = 0  # change to the GPU you want to use
IGNORE_INDEX = 255
img_idx = 10  # change to the index of the image you want to visualize
data_path = "/content/drive/MyDrive/CourseProjectAnomaly"  # drive folder of the cityscapes val set

with open("configs/dinov2/cityscapes/semantic/eomt_base_640.yaml", "r") as f:
    config_cs = yaml.safe_load(f)

with open("configs/dinov2/coco/panoptic/eomt_base_640_2x.yaml", "r") as f:
    config_coco = yaml.safe_load(f)


def create_mapping(images, ignore_index):
    unique_ids = np.unique(np.concatenate([np.unique(img) for img in images]))
    valid_ids = unique_ids[unique_ids != ignore_index]
    colors = np.array(
        [plt.cm.hsv(i / len(valid_ids))[:3] for i in range(len(valid_ids))]
    )
    mapping = {cid: colors[i] for i, cid in enumerate(valid_ids)}
    mapping[ignore_index] = np.array([0, 0, 0])
    return mapping


def apply_colormap(image, mapping):
    colored_image = np.zeros((*image.shape, 3))
    for cid in np.unique(image):
        colored_image[image == cid] = mapping.get(cid, [0, 0, 0])
    return colored_image


# Load Dataset


In [6]:
data_module_name, class_name = config_cs["data"]["class_path"].rsplit(".", 1)
data_module = getattr(importlib.import_module(data_module_name), class_name)
data_module_kwargs = config_cs["data"].get("init_args", {})

data = data_module(
    path=data_path,
    batch_size=1,
    num_workers=0,
    check_empty_targets=False,
    **data_module_kwargs
)
data.setup()

# Load model

In [7]:


# Scegliamo il config e il file dei pesi in base alla scelta sopra
use_coco = False # Metto True per il modello COCO, False per quello Cityscapes

# Scegliamo il config e il file dei pesi in base alla scelta sopra
if use_coco:
    current_config = config_coco
    state_dict_path = "/content/drive/MyDrive/CourseProjectAnomaly/eomt_coco.bin"
    target_img_size = (640, 640)
    num_classes_to_load = 133
else:
    current_config = config_cs
    state_dict_path = "/content/drive/MyDrive/CourseProjectAnomaly/eomt_cityscapes.bin"
    num_classes_to_load = data.num_classes # Usa le classi del dataset (solitamente 19 o 20)
    target_img_size = data.img_size

warnings.filterwarnings(
    "ignore",
    message=r".*Attribute 'network' is an instance of `nn\.Module` and is already saved during checkpointing.*",
)

# Load encoder
encoder_cfg = current_config["model"]["init_args"]["network"]["init_args"]["encoder"]
encoder_module_name, encoder_class_name = encoder_cfg["class_path"].rsplit(".", 1)
encoder_cls = getattr(importlib.import_module(encoder_module_name), encoder_class_name)
encoder = encoder_cls(img_size=target_img_size, **encoder_cfg.get("init_args", {}))

# Load network
network_cfg = current_config["model"]["init_args"]["network"]
network_module_name, network_class_name = network_cfg["class_path"].rsplit(".", 1)
network_cls = getattr(importlib.import_module(network_module_name), network_class_name)
network_kwargs = {k: v for k, v in network_cfg["init_args"].items() if k != "encoder"}
network = network_cls(
    masked_attn_enabled=False,
    num_classes=num_classes_to_load,
    encoder=encoder,
    **network_kwargs,
)

# Load Lightning module
lit_module_name, lit_class_name = current_config["model"]["class_path"].rsplit(".", 1)
lit_cls = getattr(importlib.import_module(lit_module_name), lit_class_name)
model_kwargs = {k: v for k, v in current_config["model"]["init_args"].items() if k != "network"}

if "stuff_classes" in current_config["data"].get("init_args", {}):
    model_kwargs["stuff_classes"] = current_config["data"]["init_args"]["stuff_classes"]

model = (
    lit_cls(
        img_size=target_img_size,
        num_classes=num_classes_to_load,
        network=network,
        **model_kwargs,
    )
    .eval()
    .to(device)
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

# Load Pretrained

In [8]:
if use_coco:
    target_img_size = (640, 640)
else:
    # Prova a prenderlo dal config, se non c'è usa quello del dataset (1024)
    target_img_size = data.img_size

model_kwargs_final = {k: v for k, v in model_kwargs.items()}

# FIX CRUCIALE: Rimuoviamo eventuali 'num_classes' dai kwargs che potrebbero sovrascrivere il nostro valore
if "num_classes" in model_kwargs_final:
  del model_kwargs_final["num_classes"]

name = current_config.get("trainer", {}).get("logger", {}).get("init_args", {}).get("name")
is_dinov3 = "dinov3" in name if name else False

if is_dinov3:
    model_kwargs["ckpt_path"] = state_dict_path
    model_kwargs["delta_weights"] = True

model = (
    lit_cls(
        img_size=target_img_size,
        num_classes=num_classes_to_load,
        network=network,
        **model_kwargs_final,
    )
    .eval()
    .to(device)
)

if not is_dinov3:
    state_dict = torch.load(
        state_dict_path, map_location=f"cuda:{device}", weights_only=True
    )
    model.load_state_dict(state_dict, strict=False)

# Semantic Inference

In [9]:
IGNORE_INDEX = 255

# DEFINIZIONE MAPPATURA UNIFICATA (COCO -> CITYSCAPES TRAIN_IDS)
# Questo dizionario mappa i category_id di COCO nei trainId di Cityscapes (0-18)
coco_to_cityscapes_map = {
    # --- THINGS (Istanze) ---
    0: 11,   # person -> person
    1: 18,   # bicycle -> bicycle
    2: 13,   # car -> car
    3: 17,   # motorcycle -> motorcycle
    5: 15,   # bus -> bus
    6: 16,   # train -> train
    7: 14,   # truck -> truck
    9: 6,    # traffic light -> traffic light
    11: 7,   # stop sign -> traffic sign (generico)

    # --- STUFF (Sfondi/Regioni) ---
    # Nota: Assicurati che questi ID corrispondano al file JSON di COCO Stuff fornito
    100: 0,  # road -> road
    123: 1,  # pavement-merged -> sidewalk
    129: 2,  # building-other-merged -> building
    91: 2,   # house -> building
    131: 3,  # wall-other-merged -> wall
    117: 4,  # fence-merged -> fence
    116: 8,  # tree-merged -> vegetation
    125: 8,  # grass-merged -> vegetation
    90: 9,   # gravel -> terrain
    102: 9,  # sand -> terrain
    119: 10, # sky-other-merged -> sky
}

def translate_preds(pred_array):
    """
    Converte i label predetti da COCO (0-132) nei label Cityscapes (0-18).
    Tutto ciò che non è nel mapping diventa IGNORE_INDEX (255).
    """
    translated = np.full_like(pred_array, fill_value=IGNORE_INDEX)
    for coco_id, city_id in coco_to_cityscapes_map.items():
        translated[pred_array == coco_id] = city_id
    return translated

def infer_semantic(img, target):
    model.eval()

    # Forza il modello a usare la dimensione della finestra corretta
    # (640 per COCO, 1024 per Cityscapes)
    model.window_size = target_img_size[0]

    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]
        crops, origins = model.window_imgs_semantic(imgs)

        mask_logits_per_layer, class_logits_per_layer = model(crops)
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], target_img_size, mode="bilinear"
        )

        crop_logits = model.to_per_pixel_logits_semantic(
            mask_logits, class_logits_per_layer[-1]
        )
        logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)
        preds = logits[0].argmax(0).cpu()

    pred_array = preds.numpy()

    # Se il modello caricato è COCO (134 classi), traduciamo i risultati
    if use_coco: # se siamo nel modello coco
        pred_array = translate_preds(pred_array)

    target_array = model.to_per_pixel_targets_semantic([target], IGNORE_INDEX)[
        0
    ].numpy()
    return pred_array, target_array


In [10]:
img_cs, target_cs = data.val_dataloader().dataset[img_idx]
pred_array_cs, target_array_cs = infer_semantic(img_cs, target_cs)

img_cc, target_cc = data.val_dataloader().dataset[img_idx]
pred_array_cc, target_array_cc = infer_semantic(img_cc, target_cc)

# Evaluation

In [11]:
import numpy as np
from tqdm import tqdm

def fast_hist(a, b, n):
    k = (a >= 0) & (a < n) & (b >= 0) & (b < n)
    return np.bincount(n * a[k].astype(int) + b[k], minlength=n**2).reshape(n, n)

def per_class_iu(hist):
    return np.diag(hist) / (hist.sum(1) + hist.sum(0) - np.diag(hist))

# setting
num_eval_classes = 19
hist = np.zeros((num_eval_classes, num_eval_classes))
val_loader = data.val_dataloader()

print(f"evaluation on {len(val_loader)} images...")
print(f"Current model: {'COCO' if use_coco else 'Cityscapes'}")

# evaluation
for i, batch in enumerate(tqdm(val_loader)):
    img = batch[0][0]
    target = batch[1][0]

    pred_array, target_array = infer_semantic(img, target)

    hist += fast_hist(target_array.flatten(), pred_array.flatten(), num_eval_classes)

# calcolo finale
ious = per_class_iu(hist)
miou = np.nanmean(ious)

# visualization results
classes = [
    "road", "sidewalk", "building", "wall", "fence", "pole", "traffic light",
    "traffic sign", "vegetation", "terrain", "sky", "person", "rider", "car",
    "truck", "bus", "train", "motorcycle", "bicycle"
]

print("\n" + "="*30)
print(f"Results MIOU - {'COCO' if use_coco else 'CITYSCAPES'}")
print("="*30)
for name, iou in zip(classes, ious):
    print(f"{name:15}: {iou*100:6.2f}%")
print("-" * 30)
print(f"MEAN IoU: {miou*100:6.2f}%")
print("="*30)

evaluation on 500 images...
Current model: Cityscapes


100%|██████████| 500/500 [06:20<00:00,  1.31it/s]


Results MIOU - CITYSCAPES
road           :  98.40%
sidewalk       :  87.36%
building       :  94.15%
wall           :  66.07%
fence          :  65.49%
pole           :  71.04%
traffic light  :  75.00%
traffic sign   :  82.13%
vegetation     :  93.02%
terrain        :  66.60%
sky            :  95.54%
person         :  85.38%
rider          :  71.11%
car            :  95.54%
truck          :  81.79%
bus            :  90.32%
train          :  77.35%
motorcycle     :  74.35%
bicycle        :  81.27%
------------------------------
MEAN IoU:  81.68%


# Parte step 7

In [12]:
seed = 42

# general reproducibility
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

NUM_CHANNELS = 3
NUM_CLASSES = 20
# gpu training specific
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

input_transform = Compose(
    [
        Resize((512, 1024), Image.BILINEAR),
        ToTensor(),
        # Normalize([.485, .456, .406], [.229, .224, .225]),
    ]
)

target_transform = Compose(
    [
        Resize((512, 1024), Image.NEAREST),
    ]
)



In [18]:
'''
parser = ArgumentParser()
parser.add_argument(
    "--input",
    default="/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly/images/*.jpg", # change the default path based on the dataset folder
    # "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly21/images/*.png"
    # "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/fs_static/images/*.jpg"
    # "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/FD_LostFound_full/images/*.png"
    # "/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadObsticle21/images/*.png"
    nargs="+",
    help="A list of space separated input images; "
    "or a single glob pattern such as 'directory/*.jpg'",
)
#parser.add_argument('--loadDir',default="../trained_models/")
#parser.add_argument('--loadWeights', default="erfnet_pretrained.pth")
#parser.add_argument('--loadModel', default="erfnet.py")
#parser.add_argument('--subset', default="val")  #can be val or train (must have labels)
#parser.add_argument('--num-workers', type=int, default=4)
#parser.add_argument('--batch-size', type=int, default=1)
#parser.add_argument('--cpu', action='store_true')
parser.add_argument('--device', type=str, default='cuda',
                    help="cpu or cuda, the device used for evaluation")
args = parser.parse_args(args=[])
anomaly_score_list_maxlogit = []
anomaly_score_list_msp = []
anomaly_score_list_maxentropy = []
anomaly_score_list_rba = []
ood_gts_list = []

path_pattern = args.input if isinstance(args.input, str) else args.input[0]
image_list = sorted(glob.glob(path_pattern))

print(f"Pattern cercato: {path_pattern}")
print(f"File trovati: {len(image_list)}")

if len(image_list) == 0:
    print("ATTENZIONE: Nessun file trovato. Verifico se le immagini hanno estensione .png...")
    image_list = sorted(glob.glob(path_pattern.replace("*.jpg", "*.png")))
    print(f"Nuovo tentativo (.png): {len(image_list)} file trovati.")


## RbA
if args.device == 'cuda' and (not torch.cuda.is_available()):
    print("Warning: Cuda is requested but cuda is not available. CPU will be used.")
    args.device = 'cpu'
DEVICE = torch.device(args.device)


def get_RbA(model, x):

    with torch.no_grad():
        out = model([{"image": x[0].to(DEVICE)}])

    if isinstance(out, dict):
        logits = out['pred_masks'].squeeze(0)
    elif isinstance(out, list) and 'pred_masks' in out[0]:
        logits = out[0]['pred_masks'].squeeze(0)
    elif isinstance(out, list) and 'sem_seg' in out[0]:
        logits = out[0]['sem_seg']

    return -logits.tanh().sum(dim=0)
##

if not os.path.exists('results.txt'):
    open('results.txt', 'w').close()
file = open('results.txt', 'a')

modelpath = args.loadDir + args.loadModel
weightspath = args.loadDir + args.loadWeights

print ("Loading model: " + modelpath)
print ("Loading weights: " + weightspath)


if (not args.cpu):
    model = torch.nn.DataParallel(model).cuda()

def load_my_state_dict(model, state_dict):  #custom function to load model when not all dict elements
    own_state = model.state_dict()
    for name, param in state_dict.items():
        if name not in own_state:
            if name.startswith("module."):
                own_state[name.split("module.")[-1]].copy_(param)
            else:
                print(name, " not loaded")
                continue
        else:
            own_state[name].copy_(param)
    return model

model = load_my_state_dict(model, torch.load(weightspath, map_location=lambda storage, loc: storage))
print ("Model and weights LOADED successfully")
model.eval()
'''


Pattern cercato: /content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly/images/*.jpg
File trovati: 60


'\n## RbA\nif args.device == \'cuda\' and (not torch.cuda.is_available()):\n    print("Warning: Cuda is requested but cuda is not available. CPU will be used.")\n    args.device = \'cpu\'\nDEVICE = torch.device(args.device)\n\n\ndef get_RbA(model, x):\n\n    with torch.no_grad():\n        out = model([{"image": x[0].to(DEVICE)}])\n\n    if isinstance(out, dict):\n        logits = out[\'pred_masks\'].squeeze(0) \n    elif isinstance(out, list) and \'pred_masks\' in out[0]:\n        logits = out[0][\'pred_masks\'].squeeze(0)\n    elif isinstance(out, list) and \'sem_seg\' in out[0]:\n        logits = out[0][\'sem_seg\']\n\n    return -logits.tanh().sum(dim=0)\n##\n\nif not os.path.exists(\'results.txt\'):\n    open(\'results.txt\', \'w\').close()\nfile = open(\'results.txt\', \'a\')\n\nmodelpath = args.loadDir + args.loadModel\nweightspath = args.loadDir + args.loadWeights\n\nprint ("Loading model: " + modelpath)\nprint ("Loading weights: " + weightspath)\n\n\nif (not args.cpu):\n    mod

In [28]:
import os
import glob
import torch
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from PIL import Image
from argparse import ArgumentParser

# 1. DEFINIZIONE ARGS (Se sono sottolineati, dobbiamo definirli qui)
parser = ArgumentParser()
parser.add_argument(
    "--input",
    default="/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly/images/*.jpg",
    nargs="+",
    help="Percorso delle immagini"
)
# Usiamo parse_known_args() per evitare conflitti con i parametri interni di Colab
args, _ = parser.parse_known_args()

# 2. CONFIGURAZIONE E LISTE
# Assicurati che target_img_size sia definita (es. 1024 o 640)
target_img_size = (1024, 1024)
print(f"Usando target_size: {target_img_size}")

anomaly_score_list_maxlogit = []
anomaly_score_list_msp = []
anomaly_score_list_maxentropy = []
anomaly_score_list_rba = []

# Risoluzione path sottolineato
path_pattern = args.input[0] if isinstance(args.input, list) else args.input
image_list = sorted(glob.glob(path_pattern))

print(f"Inizio valutazione su {len(image_list)} immagini...")

# 3. LOOP DI VALUTAZIONE
model.eval()
device = next(model.parameters()).device
model_ref = model.module if isinstance(model, torch.nn.DataParallel) else model

for img_path in tqdm(image_list):
    # Caricamento immagine
    img_pil = Image.open(img_path).convert('RGB')

    # Versione normalizzata per il modello (float)
    img_tensor_norm = input_transform(img_pil).unsqueeze(0).to(device)

    # Versione RAW (uint8) per window_imgs_semantic (evita l'errore PIL visto prima)
    img_tensor_raw = torch.from_numpy(np.array(img_pil)).permute(2, 0, 1).to(device)

    with torch.no_grad():
        # Usiamo la forma più stabile di autocast per la tua versione di PyTorch
        with torch.cuda.amp.autocast(enabled=True):
            img_sizes = [img_tensor_norm.shape[-2:]]

            # Windowing con il tensore RAW
            crops, origins = model_ref.window_imgs_semantic([img_tensor_raw])

            # Forward pass
            mask_logits_per_layer, class_logits_per_layer = model(crops)

            # Interpolazione all'ultimo layer
            mask_logits = F.interpolate(
                mask_logits_per_layer[-1],
                size=target_img_size,
                mode="bilinear",
                align_corners=False
            )

            # Conversione in logit per pixel
            crop_logits = model_ref.to_per_pixel_logits_semantic(
                mask_logits, class_logits_per_layer[-1]
            )

            # Ricomposizione
            logits = model_ref.revert_window_logits_semantic(crop_logits, origins, img_sizes)[0]

    # --- CALCOLO SCORE ---
    max_logit, _ = torch.max(logits, dim=0)
    anomaly_score_list_maxlogit.append((-max_logit).cpu().numpy())

    probs = torch.softmax(logits, dim=0)
    msp, _ = torch.max(probs, dim=0)
    anomaly_score_list_msp.append((1.0 - msp).cpu().numpy())

    entropy = -torch.sum(probs * torch.log(probs + 1e-7), dim=0)
    anomaly_score_list_maxentropy.append(entropy.cpu().numpy())

    rba = -logits.tanh().sum(dim=0)
    anomaly_score_list_rba.append(rba.cpu().numpy())

print("\nInferenza completata con successo!")

Usando target_size: (1024, 1024)
Inizio valutazione su 60 immagini...


  0%|          | 0/60 [00:00<?, ?it/s]/tmp/ipykernel_13875/146747393.py:54: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=True):
100%|██████████| 60/60 [01:19<00:00,  1.33s/it]


Inferenza completata con successo!


In [38]:
import numpy as np
from PIL import Image
from sklearn.metrics import average_precision_score, roc_curve

# 1. DEFINIZIONE DI ood_gts (Se non è già un array, lo creiamo dalla lista)
# Assicurati di aver eseguito la cella dove carichi ood_gts_list
ood_gts = np.array(ood_gts_list)

# 2. Rileviamo la dimensione reale dei tuoi logit (es. 512x1024)
current_logit_size = (anomaly_score_list_maxlogit[0].shape[0], anomaly_score_list_maxlogit[0].shape[1])
print(f"Dimensioni logit modello: {current_logit_size}")

# 3. Ridimensioniamo le maschere per farle combaciare
ood_gts_final = []
for gt in ood_gts:
    gt_pil = Image.fromarray(gt)
    # Ridimensioniamo a 1024x512 (attenzione all'ordine Width, Height in PIL)
    gt_res = gt_pil.resize((current_logit_size[1], current_logit_size[0]), Image.NEAREST)
    ood_gts_final.append(np.array(gt_res))

ood_gts_final = np.array(ood_gts_final)

# 4. Creiamo le maschere booleane
ood_mask = (ood_gts_final == 1)
ind_mask = (ood_gts_final == 0)

# 5. Funzione per FPR@95
def fpr_at_95_tpr(conf, labels):
    if len(np.unique(labels)) < 2: return 0.0
    fpr, tpr, _ = roc_curve(labels, conf)
    return fpr[np.where(tpr >= 0.95)[0][0]]

# 6. Dizionario dei metodi e calcolo metriche
anomaly_scores_dict = {
    "maxlogit": np.array(anomaly_score_list_maxlogit),
    "msp": np.array(anomaly_score_list_msp),
    "maxentropy": np.array(anomaly_score_list_maxentropy),
    "rba": np.array(anomaly_score_list_rba)
}

file = open('results.txt', 'a')
file.write(f"\n--- Valutazione RoadAnomaly (Size: {current_logit_size}) ---\n")

for name, scores in anomaly_scores_dict.items():
    # Pulizia NaN (fondamentale per evitare ValueError)
    if np.isnan(scores).any():
        print(f"Attenzione: rilevati NaN in {name}, sostituiti con 0.0")
        scores = np.nan_to_num(scores, nan=0.0)

    # Estrazione pixel
    ood_out = scores[ood_mask]
    ind_out = scores[ind_mask]

    val_out = np.concatenate((ind_out, ood_out))
    val_label = np.concatenate((np.zeros(len(ind_out)), np.ones(len(ood_out))))

    # Calcolo metriche reali
    prc_auc = average_precision_score(val_label, val_out)
    fpr = fpr_at_95_tpr(val_out, val_label)

    print(f'AUPRC score {name}: {prc_auc*100.0:.4f}')
    print(f'FPR@TPR95 {name}: {fpr*100.0:.4f}')
    print("-" * 20)

    file.write(f'AUPRC {name}: {prc_auc*100.0:.4f} | FPR@95 {name}: {fpr*100.0:.4f}\n')

file.close()
print("Processo completato con successo!")

Dimensioni logit modello: (512, 1024)
Attenzione: rilevati NaN in maxlogit, sostituiti con 0.0
AUPRC score maxlogit: 17.4160
FPR@TPR95 maxlogit: 81.0496
--------------------
Attenzione: rilevati NaN in msp, sostituiti con 0.0
AUPRC score msp: 43.0152
FPR@TPR95 msp: 73.4393
--------------------
Attenzione: rilevati NaN in maxentropy, sostituiti con 0.0
AUPRC score maxentropy: 42.6913
FPR@TPR95 maxentropy: 74.2192
--------------------
Attenzione: rilevati NaN in rba, sostituiti con 0.0
AUPRC score rba: 17.1509
FPR@TPR95 rba: 91.4382
--------------------
Processo completato con successo!


## da qui in giù boh, era la parte prima che non ho modificato

In [19]:
def get_anomaly_scores_eomt(model, img_tensor, target_size):
    model.eval()
    device = next(model.parameters()).device

    # Prepara l'immagine
    if len(img_tensor.shape) == 3:
        img_tensor = img_tensor.unsqueeze(0)
    img_tensor = img_tensor.to(device)

    with torch.no_grad(), autocast(device_type="cuda", dtype=torch.float16):
        # Utilizziamo la logica di windowing specifica del tuo modello EoMT
        img_sizes = [img_tensor.shape[-2:]]
        # model.window_imgs_semantic si aspetta una lista di immagini
        crops, origins = model.window_imgs_semantic([img_tensor[0]])

        # Forward pass
        mask_logits_per_layer, class_logits_per_layer = model(crops)

        # Interpolazione e proiezione per pixel (usiamo l'ultimo layer [-1])
        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], target_size, mode="bilinear", align_corners=False
        )

        crop_logits = model.to_per_pixel_logits_semantic(
            mask_logits, class_logits_per_layer[-1]
        )

        # Ricostruzione dell'immagine intera dai vari crop (windows)
        logits = model.revert_window_logits_semantic(crop_logits, origins, img_sizes)[0]
        # logits ora è [Num_Classes, H, W]

    # --- CALCOLO METRICHE POST-HOC ---

    # A. Max Logit (Invertito: valori alti = più anomalo)
    max_logit, _ = torch.max(logits, dim=0)
    score_maxlogit = -max_logit

    # B. MSP (Maximum Softmax Probability)
    probs = torch.softmax(logits, dim=0)
    msp, _ = torch.max(probs, dim=0)
    score_msp = 1.0 - msp

    # C. Max Entropy
    entropy = -torch.sum(probs * torch.log(probs + 1e-7), dim=0)
    score_entropy = entropy

    # D. RbA (Region-based Anomaly) - Specifica per Mask Architectures
    score_rba = -logits.tanh().sum(dim=0)

    return (score_maxlogit.cpu().numpy(),
            score_msp.cpu().numpy(),
            score_entropy.cpu().numpy(),
            score_rba.cpu().numpy())

anomaly_score_list_maxlogit = []
anomaly_score_list_msp = []
anomaly_score_list_maxentropy = []
anomaly_score_list_rba = []
ood_gts_list = []

# Recuperiamo la lista delle immagini dal parser (già definito prima)
image_list = sorted(glob.glob(args.input[0]))
print(f"Trovate {len(image_list)} immagini per la validazione anomalie.")

# 3. Loop di Inferenza
for img_path in tqdm(image_list):
    # Caricamento e trasformazione
    img_pil = Image.open(img_path).convert('RGB')
    img_tensor = input_transform(img_pil)

    # Eseguiamo l'inferenza
    # target_img_size deve essere (640, 640) per COCO o quello del dataset per Cityscapes
    s_ml, s_msp, s_ent, s_rba = get_anomaly_scores_eomt(model, img_tensor, target_img_size)

    # Salvataggio risultati
    anomaly_score_list_maxlogit.append(s_ml)
    anomaly_score_list_msp.append(s_msp)
    anomaly_score_list_maxentropy.append(s_ent)
    anomaly_score_list_rba.append(s_rba)

    # Caricamento Ground Truth (opzionale, se hai i file .png delle maschere di anomalia)
    # Esempio: gt_path = img_path.replace("images", "labels").replace(".jpg", ".png")
    # gt = np.array(Image.open(gt_path))
    # ood_gts_list.append(gt)

print("\nInferenza completata per tutti i metodi (MSP, Max Logit, Entropy, RbA).")


Trovate 1 immagini per la validazione anomalie.


  0%|          | 0/1 [00:00<?, ?it/s]


IsADirectoryError: [Errno 21] Is a directory: '/'

In [ ]:
'''
# Controlliamo se args.input è una lista o una stringa
input_pattern = args.input[0] if isinstance(args.input, list) else args.input

for path in glob.glob(os.path.expanduser(str(input_pattern))):
    print(path)
    images = input_transform((Image.open(path).convert('RGB'))).unsqueeze(0).float().cuda()
    #images = images.permute(0,3,1,2)
    with torch.no_grad():
        #result = model(images)
        result_raw = model([{"image": images[0]}])

        pred_logits = result_raw[0]['pred_logits']
        pred_masks = result_raw[0]['pred_masks']

        mask_probs = pred_masks.sigmoid() # [Queries, H, W]

        result = torch.einsum("qc,qhw->chw", pred_logits, mask_probs)

    #compute max logit
    anomaly_result_maxlogit = 1.0 - np.max(result.squeeze(0).data.cpu().numpy(), axis=0)

    #compute MSP
    probs = torch.nn.functional.softmax(result, dim=1)
    max_probs, _ = torch.max(probs.squeeze(0), dim=0)
    anomaly_result_msp = 1.0 - max_probs.data.cpu().numpy()

    #compute Max Entropy
    probs = torch.nn.functional.softmax(result, dim=1).squeeze(0)
    epsilon = 1e-10
    entropy = -torch.sum(probs * torch.log(probs + epsilon), dim=0)
    anomaly_result_maxentropy = entropy.data.cpu().numpy()

    #compute RbA
    anomaly_result_RbA = get_RbA(model, images).data.cpu().numpy()

    pathGT = path.replace("images", "labels_masks")
    if "RoadObsticle21" in pathGT:
        pathGT = pathGT.replace("webp", "png")
    if "fs_static" in pathGT:
        pathGT = pathGT.replace("jpg", "png")
    if "RoadAnomaly" in pathGT:
        pathGT = pathGT.replace("jpg", "png")

    mask = Image.open(pathGT)
    mask = target_transform(mask)
    ood_gts = np.array(mask)



    if "RoadAnomaly" in pathGT:
      #in RoadAnomaly 2 indica anomalia --> viene trasformato in 1 per uniformare
        ood_gts = np.where((ood_gts==2), 1, ood_gts)

    if 1 not in np.unique(ood_gts):
        continue   #se non c'è nessuna anomalia, allora salta l'immagine e passa a valutare la successiva
    else:
          ood_gts_list.append(ood_gts)
          anomaly_score_list_maxlogit.append(anomaly_result_maxlogit)
          anomaly_score_list_msp.append(anomaly_result_msp)
          anomaly_score_list_maxentropy.append(anomaly_result_maxentropy)
          anomaly_score_list_rba.append(anomaly_result_RbA)

    del result, anomaly_result_maxlogit,anomaly_result_msp, anomaly_result_maxentropy, ood_gts, mask
    torch.cuda.empty_cache()

file.write( "------\n")'''

/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly/images/48.jpg
/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly/images/33.jpg
/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly/images/28.jpg
/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly/images/31.jpg
/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly/images/26.jpg
/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly/images/16.jpg
/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly/images/10.jpg
/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_Datasets/Validation_Dataset/RoadAnomaly/images/55.jpg
/content/drive/MyDrive/CourseProjectAnomaly/Anomaly_Validation_D

7

In [ ]:
ood_gts = np.array(ood_gts_list)
anomaly_scores_maxlogit = np.array(anomaly_score_list_maxlogit)
anomaly_scores_msp = np.array(anomaly_score_list_msp)
anomaly_scores_maxentropy = np.array(anomaly_score_list_maxentropy)
anomaly_scores_nba = np.array(anomaly_score_list_nba)

ood_mask = (ood_gts == 1)
ind_mask = (ood_gts == 0)

# max logit
ood_out = anomaly_scores_maxlogit[ood_mask]
ind_out = anomaly_scores_maxlogit[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score maxlogit: {prc_auc*100.0}')
print(f'FPR@TPR95 maxlogit: {fpr*100.0}')

file.write(('    AUPRC score maxlogit:' + str(prc_auc*100.0) + '   FPR@TPR95 maxlogit:' + str(fpr*100.0) ))

#MSP
ood_out = anomaly_scores_msp[ood_mask]
ind_out = anomaly_scores_msp[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score msp: {prc_auc*100.0}')
print(f'FPR@TPR95 msp: {fpr*100.0}')

file.write(('    AUPRC score msp:' + str(prc_auc*100.0) + '   FPR@TPR95 msp:' + str(fpr*100.0) ))

#max entropy
ood_out = anomaly_scores_maxentropy[ood_mask]
ind_out = anomaly_scores_maxentropy[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score maxentropy: {prc_auc*100.0}')
print(f'FPR@TPR95 maxentropy: {fpr*100.0}')

file.write(('    AUPRC score maxentropy:' + str(prc_auc*100.0) + '   FPR@TPR95 maxentropy:' + str(fpr*100.0) ))

# rba
ood_out = anomaly_scores_rba[ood_mask]
ind_out = anomaly_scores_rba[ind_mask]

ood_label = np.ones(len(ood_out))
ind_label = np.zeros(len(ind_out))

val_out = np.concatenate((ind_out, ood_out))
val_label = np.concatenate((ind_label, ood_label))

prc_auc = average_precision_score(val_label, val_out)
fpr = fpr_at_95_tpr(val_out, val_label)

print(f'AUPRC score rba: {prc_auc*100.0}')
print(f'FPR@TPR95 rba: {fpr*100.0}')

file.write(('    AUPRC score rba:' + str(prc_auc*100.0) + '   FPR@TPR95 rba:' + str(fpr*100.0) ))

file.close()

AUPRC score maxlogit: 15.581983299743523
FPR@TPR95 maxlogit: 73.24763926329555
AUPRC score msp: 12.422651180579319
FPR@TPR95 msp: 82.5754588847072
AUPRC score maxentropy: 12.668527734637614
FPR@TPR95 maxentropy: 82.74862138731258


# Evaluation mean Intersection Over Union

In [ ]:
# Crea la cartella di destinazione
!mkdir -p /content/cityscapes_data

# Unzip delle immagini (usa il nome esatto del tuo file zip)
!unzip -q /content/drive/MyDrive/CourseProjectAnomaly/cityscapes/leftImg8bit_trainvaltest1.zip -d /content/cityscapes_data/

# Unzip delle maschere
!unzip -q /content/drive/MyDrive/CourseProjectAnomaly/cityscapes/gtFine_trainvaltest1.zip -d /content/cityscapes_data/

replace /content/cityscapes_data/README? [y]es, [n]o, [A]ll, [N]one, [r]ename: A


In [ ]:
# 1. Installiamo gli script ufficiali di Cityscapes
!pip install cityscapesscripts

# 2. Diciamo agli script dove si trova la nostra cartella scompattata
import os
os.environ['CITYSCAPES_DATASET'] = '/content/cityscapes_data/'

# 3. Lanciamo il convertitore ufficiale
!csCreateTrainIdLabelImgs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 5.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 473.6/473.6 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 10.4 MB/s eta 0:00:00
  Created wheel for typing: filename=typing-3.7.4.3-py3-none-any.whl size=26304 sha256=9550af8f6d5054904eaf500e8c3148b82291d4402f33003805aa35a582d92b14
  Stored in directory: /root/.cache/pip/wheels/12/98/52/2bffe242a9a487f00886e43b8ed8dac46456702e11a0d6abef
Successfully built typing


Processing 5000 annotation files
Progress: 100.0 % 

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import os
import importlib
import time

from PIL import Image
from argparse import ArgumentParser

from torch.autograd import Variable
from torch.utils.data import DataLoader
from torchvision.transforms import Compose, CenterCrop, Normalize, Resize
from torchvision.transforms import ToTensor, ToPILImage

from dataset import cityscapes
from erfnet import ERFNet
from transform import Relabel, ToLabel, Colorize
from iouEval import iouEval, getColorEntry

image_transform = ToPILImage()
input_transform_cityscapes = Compose([
    Resize(512, Image.BILINEAR),
    ToTensor(),
])
target_transform_cityscapes = Compose([
    Resize(512, Image.NEAREST),
    ToLabel(),
    Relabel(255, 19),   #ignore label to 19
])


parser = ArgumentParser()

parser.add_argument('--state')

parser.add_argument('--loadDir',default="../trained_models/")
parser.add_argument('--loadWeights', default="erfnet_pretrained.pth")
parser.add_argument('--loadModel', default="erfnet.py")
parser.add_argument('--subset', default="val")  #can be val or train (must have labels)
parser.add_argument('--datadir', default="/content/cityscapes_data/")
parser.add_argument('--num-workers', type=int, default=2) # diminuito da 4 a 2
parser.add_argument('--batch-size', type=int, default=1)
parser.add_argument('--cpu', action='store_true')
args = parser.parse_args(args=[])


if(not os.path.exists(args.datadir)):
    print ("Error: datadir could not be loaded")


loader = DataLoader(cityscapes(args.datadir, input_transform_cityscapes, target_transform_cityscapes, subset=args.subset), num_workers=args.num_workers, batch_size=args.batch_size, shuffle=False)


iouEvalVal = iouEval(NUM_CLASSES)

start = time.time()

for step, (images, labels, filename, filenameGt) in enumerate(loader):
    if (not args.cpu):
        images = images.cuda()
        labels = labels.cuda()

    inputs = Variable(images)
    with torch.no_grad():
        outputs = model(inputs)

    iouEvalVal.addBatch(outputs.max(1)[1].unsqueeze(1).data, labels)

    filenameSave = filename[0].split("leftImg8bit/")[1]

    print (step, filenameSave)


iouVal, iou_classes = iouEvalVal.getIoU()

iou_classes_str = []
for i in range(iou_classes.size(0)):
    iouStr = getColorEntry(iou_classes[i])+'{:0.2f}'.format(iou_classes[i]*100) + '\033[0m'
    iou_classes_str.append(iouStr)

print("---------------------------------------")
print("Took ", time.time()-start, "seconds")
print("=======================================")
#print("TOTAL IOU: ", iou * 100, "%")
print("Per-Class IoU:")
print(iou_classes_str[0], "Road")
print(iou_classes_str[1], "sidewalk")
print(iou_classes_str[2], "building")
print(iou_classes_str[3], "wall")
print(iou_classes_str[4], "fence")
print(iou_classes_str[5], "pole")
print(iou_classes_str[6], "traffic light")
print(iou_classes_str[7], "traffic sign")
print(iou_classes_str[8], "vegetation")
print(iou_classes_str[9], "terrain")
print(iou_classes_str[10], "sky")
print(iou_classes_str[11], "person")
print(iou_classes_str[12], "rider")
print(iou_classes_str[13], "car")
print(iou_classes_str[14], "truck")
print(iou_classes_str[15], "bus")
print(iou_classes_str[16], "train")
print(iou_classes_str[17], "motorcycle")
print(iou_classes_str[18], "bicycle")
print("=======================================")
iouStr = getColorEntry(iouVal)+'{:0.2f}'.format(iouVal*100) + '\033[0m'
print ("MEAN IoU: ", iouStr, "%")


/content/cityscapes_data/leftImg8bit/val /content/cityscapes_data/gtFine/val
0 val/frankfurt/frankfurt_000000_000294_leftImg8bit.png
1 val/frankfurt/frankfurt_000000_000576_leftImg8bit.png
2 val/frankfurt/frankfurt_000000_001016_leftImg8bit.png
3 val/frankfurt/frankfurt_000000_001236_leftImg8bit.png
4 val/frankfurt/frankfurt_000000_001751_leftImg8bit.png
5 val/frankfurt/frankfurt_000000_002196_leftImg8bit.png
6 val/frankfurt/frankfurt_000000_002963_leftImg8bit.png
7 val/frankfurt/frankfurt_000000_003025_leftImg8bit.png
8 val/frankfurt/frankfurt_000000_003357_leftImg8bit.png
9 val/frankfurt/frankfurt_000000_003920_leftImg8bit.png
10 val/frankfurt/frankfurt_000000_004617_leftImg8bit.png
11 val/frankfurt/frankfurt_000000_005543_leftImg8bit.png
12 val/frankfurt/frankfurt_000000_005898_leftImg8bit.png
13 val/frankfurt/frankfurt_000000_006589_leftImg8bit.png
14 val/frankfurt/frankfurt_000000_007365_leftImg8bit.png
15 val/frankfurt/frankfurt_000000_008206_leftImg8bit.png
16 val/frankfurt/fran